In [1]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from datetime import date
import re
from tqdm import tqdm
import time
from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr
from scipy.spatial.distance import jaccard, euclidean, cosine

import functools
import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, nan_euclidean_distances

today = date.today()

# Create a "recipe" that gives all the attributes of a region (regardless of region size).

* SNP
* Transcription Factor
* DHS
* Chromatin State
* Genes

Currently only 25kb regions has been worked on. This file utilizes and works with the 'create_combined_score' notebook.

## Read in all related dataframes

In [2]:
# 25kb bins
bin_cols = ['seqname', 'start', 'end', '?']
bins_25kb = pd.read_csv('../data/Altius_100k_epilogos_hg19_WM20210518.bed', delimiter='\t', names=bin_cols)
print(bins_25kb)


# SNP data
snp_cols = ['Chromosome', 'p1', 'p2', 'Reference SNP cluster ID', 'pvalue', 'phenotype']
snp_df = pd.read_csv('../data/UKBB_all_traits_all_sig_associations_CT20220207.bed', delimiter='\t', names=snp_cols)
print(snp_df)

trait_codes = []
with gzip.open('/home/nasi4/proj/GWAS/UKBB_201807/info/phenotypes.both_sexes.tsv.gz','r') as f:
    for line in f:
        trait_codes.append(line.decode("utf-8").strip('\n').split('\t'))

trait_codes_df = pd.DataFrame(trait_codes[1:], columns=trait_codes[0])
print(trait_codes_df)

# TF data 
# NOTE: THIS IS CURRENTLY HG38
tf_pd = pd.read_pickle('../data/TF-df-hg38.pkl')
print(tf_pd)

# DHS data
dhs_df = pd.read_pickle('../data/dhs_metadata.pkl')
print(dhs_df)

dhs_Mixture_df = pd.read_pickle('../data/dhs_mixture_df.pkl')
print(dhs_Mixture_df)

dhs_obs_over_exp = pd.read_pickle('../data/dhs_obs_over_exp_25kb.pkl')
print(dhs_obs_over_exp)

# Chromatin State Data
epilogos_833_df = pd.read_pickle('../data/epilogos_state_data_18.pkl')
print(epilogos_833_df)

epilogos_states_df = pd.read_pickle('../data/epilogos_states_df')
print(epilogos_states_df)

epilogos_metadata = pd.read_csv('../data/All_833_biosamples/metadata.tsv', delimiter='\t')
print(epilogos_metadata)


# Gene Data
string_assoc = pd.read_pickle('../data/hg19-protein-associations.pkl')
print(string_assoc)

gene_hg19_bed = pd.read_pickle('../data/hg19_gene_bed.pkl')
print(gene_hg19_bed)

protein_assoc_hg19 = pd.read_pickle('../data/hg19-protein-associations-detailed.pkl')
print(protein_assoc_hg19)

      seqname      start        end     ?
0        chr1       9000      34000  0.41
1        chr1      34000      59000  0.41
2        chr1      61000      86000  0.41
3        chr1      86000     111000  0.41
4        chr1     111000     136000  0.41
...       ...        ...        ...   ...
99991    chrX  155042800  155067800  0.41
99992    chrX  155067800  155092800  0.41
99993    chrX  155092800  155117800  0.41
99994    chrX  155117800  155142800  0.41
99995    chrX  155142800  155167800  0.41

[99996 rows x 4 columns]
         Chromosome         p1         p2 Reference SNP cluster ID  \
0             chr12  130300801  130300802               rs73150827   
1              chr4   99318162   99318163                rs1229984   
2              chr9    8303561    8303562              rs112120933   
3              chr1   46941503   46941504              rs141586057   
4             chr20   52932649   52932650              rs185961463   
...             ...        ...        ...         

identifier   1.10011   1.10021   1.10025   1.10027    1.1003   1.10032  \
0           0.001439  0.000000  0.000000  0.001336  0.000000  0.000000   
1           0.000000  0.000000  0.000000  0.001010  0.000000  0.000000   
2           0.000000  0.000000  0.000000  0.011659  0.000893  0.000000   
3           0.000000  0.000311  0.000000  0.008287  0.000000  0.000000   
4           0.000204  0.000000  0.000000  0.004569  0.000134  0.001297   
5           0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
6           0.000040  0.011431  0.000000  0.002904  0.006965  0.000000   
7           0.000000  0.000000  0.025745  0.001445  0.000000  0.000000   
8           0.000000  0.000000  0.000000  0.000000  0.000208  0.000000   
9           0.000000  0.000000  0.000000  0.000000  0.001768  0.000000   
10          0.000000  0.000000  0.000000  0.000000  0.003912  0.000000   
11          0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
12          0.000000  0.000000  0.0000

         seqname      start        end   0   1   2   3   4   5   6  ...  823  \
0           chr1          0        200  18  18  18  18  18  18  18  ...   18   
1           chr1        200        400  18  18  18  18  18  18  18  ...   18   
2           chr1        400        600  18  18  18  18  18  18  18  ...   18   
3           chr1        600        800  18  18  18  18  18  18  18  ...   18   
4           chr1        800       1000  18  18  18  18  18  18  18  ...   18   
...          ...        ...        ...  ..  ..  ..  ..  ..  ..  ..  ...  ...   
15181503    chrX  155269400  155269600  18  18  18  18  18  18  18  ...   18   
15181504    chrX  155269600  155269800  18  18  18  18  18  18  18  ...   18   
15181505    chrX  155269800  155270000  18  18  18  18  18  18  18  ...   18   
15181506    chrX  155270000  155270200  18  18  18  18  18  18  18  ...   18   
15181507    chrX  155270200  155270400  18  18  18  18  18  18  18  ...   18   

          824  825  826  827  828  829 

              chr      start        end   4 strand                6      7  \
gene_names                                                                   
ARF5         chr7  127228398  127231759   6      +  ENST00000000233    658   
M6PR        chr12    9068355    9094058  17      +  ENST00000536844   6752   
FKBP4       chr12    2904118    2913124  46      +  ENST00000001008   3720   
CYP26B1      chr2   72356366   72375167  26      -  ENST00000001146  20581   
NDUFAF7      chr2   37458805   37476303   5      +  ENST00000474154  28816   
...           ...        ...        ...  ..    ...              ...    ...   
CERS1       chr19   18988719   19007536   8      -  ENST00000542296  14253   
ZNF518A     chr10   97889828   97965044   0      +  ENST00000563195  29009   
SPATA31A5    chr9   90529407   90538572   0      +  ENST00000437823  32005   
EIF3L       chr22   38244874   38285414   1      +  ENST00000412331  18138   
OR6Q1       chr11   57798401   57799403   0      +  ENST00000302

In [23]:
### Get GATA1's closest 25kb region. ###

gene_hg19_bed = pd.read_pickle('../data/hg19_gene_bed.pkl')
bin_cols = ['seqname', 'start', 'end', '?']
bins_25kb = pd.read_csv('../data/Altius_100k_epilogos_hg19_WM20210518.bed', delimiter='\t', names=bin_cols)

# Use Specific Gene's closest regions to their start and end
gene = 'GATA1'

gene_chr = gene_hg19_bed.loc[gene].chr
gene_start = gene_hg19_bed.loc[gene].start
gene_end = gene_hg19_bed.loc[gene].end

print(gene_hg19_bed.loc[gene])
print(gene_start)
print(gene_end)


## Second half of GATA1 region is chosen instead of the first half. You can uncomment to switch.
# region = bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['start'] <= gene_start)].iloc[-1]
region = bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['end'] >= gene_end)].iloc[0]
# bins_25kb
region

chr                                  chrX
start                            48644961
end                              48652715
4                                     200
strand                                  +
6                         ENST00000376670
7                                    4170
8                          protein-coding
descriptor         GATA binding protein 1
10                               48649516
11                               48652571
12                92,239,378,146,126,516,
13            0,4536,5289,5768,6617,7238,
Name: GATA1, dtype: object
48644961
48652715


seqname        chrX
start      48647800
end        48672800
?           7.49905
Name: 96548, dtype: object

In [56]:
### USING A RANDOM REGION ###
rand_row = random.randint(0, len(bins_25kb))
rand_row
region = bins_25kb.iloc[rand_row]
print(region)

seqname        chr8
start      65632200
end        65657200
?           5.15043
Name: 87637, dtype: object


In [3]:
### USE SPECIFIC REGION ###
# 62105 is the top region for the GATA1 region (96548) for combined_score (0.9999999472885749) 
# - percentile of cosine similarity. Where Combined Score is comprised of TF, DHS, and epilogos

# Current results on this notebook are that 57993 is the top region for GATA1 with STRING included in the 
# Combined Score but I believe it was calculated incorrectly in the create_combined_score notebook.

region = bins_25kb.iloc[57993, :]

# 25kb 

In [14]:
# Use Specific Gene's closest regions
gene = 'GATA1'

gene_chr = gene_hg19_bed.loc[gene].chr
gene_start = gene_hg19_bed.loc[gene].start
gene_end = gene_hg19_bed.loc[gene].end

print(gene_hg19_bed.loc[gene])
print(gene_start)
print(gene_end)
# print(bins_25kb[bins_25kb['seqname']==gene_chr].loc[96548])

print(bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['start'] <= gene_start)].iloc[-1])
print(bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['end'] >= gene_end)].iloc[0])

# region = bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['start'] <= gene_start)].iloc[-1]
region = bins_25kb.loc[(bins_25kb['seqname'] == gene_chr) & (bins_25kb['end'] >= gene_end)].iloc[0]

chr                                  chrX
start                            48644961
end                              48652715
4                                     200
strand                                  +
6                         ENST00000376670
7                                    4170
8                          protein-coding
descriptor         GATA binding protein 1
10                               48649516
11                               48652571
12                92,239,378,146,126,516,
13            0,4536,5289,5768,6617,7238,
Name: GATA1, dtype: object
48644961
48652715
seqname        chrX
start      48583800
end        48608800
?           7.49548
Name: 96547, dtype: object
seqname        chrX
start      48647800
end        48672800
?           7.49905
Name: 96548, dtype: object


In [4]:
region_index = region.name
region_chr = region.seqname
region_start = region.start
region_end = region.end
print(region_chr, region_start, region_end)

chr3 57729600 57754600


In [5]:
### SNP ###

trait_codes = []
with gzip.open('/home/nasi4/proj/GWAS/UKBB_201807/info/phenotypes.both_sexes.tsv.gz','r') as f:
    for line in f:
        trait_codes.append(line.decode("utf-8").strip('\n').split('\t'))

trait_codes_df = pd.DataFrame(trait_codes[1:], columns=trait_codes[0])
print(trait_codes_df)

snp_chr = snp_df.loc[snp_df['Chromosome'] == region_chr]
snp_in_region = snp_chr[(snp_chr['p1'] >= region_start) & (snp_chr['p2'] <= region_end)]

trait_codes_temp = trait_codes_df.set_index(trait_codes_df['phenotype'])

snp_in_region['description'] = snp_in_region['phenotype'].map(trait_codes_temp['description'])
snp_in_region.sort_values(by='pvalue')['description']

       phenotype                                        description  \
0            G54  Diagnoses - main ICD10: G54 Nerve root and ple...   
1            C21  Diagnoses - main ICD10: C21 Malignant neoplasm...   
2            F45   Diagnoses - main ICD10: F45 Somatoform disorders   
3            I35  Diagnoses - main ICD10: I35 Nonrheumatic aorti...   
4            D38  Diagnoses - main ICD10: D38 Neoplasm of uncert...   
...          ...                                                ...   
4354  100023_raw                                             Starch   
4355  100024_raw                                            Calcium   
4356  100025_raw                                          Vitamin E   
4357   is_female           is_female, based on inferred genetic sex   
4358         age                                 Age at recruitment   

       variable_type     source n_non_missing n_missing n_controls n_cases  \
0        categorical      icd10        361194         0     361051   

/tmp/ipykernel_81848/1661537897.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  snp_in_region['description'] = snp_in_region['phenotype'].map(trait_codes_temp['description'])


8887588                   Mean platelet (thrombocyte) volume
8887599                   Mean platelet (thrombocyte) volume
8887591                   Mean platelet (thrombocyte) volume
8887590                   Mean platelet (thrombocyte) volume
8887596                   Mean platelet (thrombocyte) volume
                                 ...                        
2933227                             Job SOC coding: Printers
4152574    Diagnoses - main ICD10: E23 Hypofunction and o...
9219321                                    Neutrophill count
8887597                   Mean platelet (thrombocyte) volume
5632142    Forced expiratory volume in 1-second (FEV1), B...
Name: description, Length: 74, dtype: object

In [6]:
### TF ### - Gives the best TF motif for a 25kb region based on highest enrichment score.
# motif_counts_df = pd.read_pickle('../data/motif_counts_df_25kb.pkl')
# motif_counts_df.iloc[:, 3:] = motif_counts_df.iloc[:, 3:].astype(int)

# # motif_counts_chr = motif_counts_df[motif_counts_df['seqname'] == region_chr]
# # motif_sub = motif_counts_chr.loc[(motif_counts_chr['start'] >= region_start) & (motif_counts_chr['end'] <= region_end)] 
# # print(motif_sub)


# total_tf_cluster_counts_expected = motif_counts_df.iloc[:, 3:].sum(axis=0)/len(bins_25kb)
# # print(motif_sub.iloc[:, 3:])

# # (np.log2(motif_sub.iloc[:, 3:]/total_tf_cluster_counts_expected)).T

# motif_enrichment = (np.log2(motif_counts_df.iloc[:, 3:]/total_tf_cluster_counts_expected))
# print(motif_enrichment)
# print(motif_counts_df.iloc[:, :3])

# motif_enrichment = pd.concat([motif_counts_df.iloc[:, :3], motif_enrichment], axis=1)
# print(motif_enrichment)

# motif_enrichment.to_pickle('../data/motif_log2_enrichment.pkl')

# This was calculated through dividing counts found through bedmap by expected counts if all 
# TF motifs were divided evenly throughout the genome
motif_obs_over_exp = pd.read_pickle('../data/motif_obs_over_exp.pkl')

motif_sub = motif_obs_over_exp.iloc[region_index, :]
print(motif_sub.idxmax())
print(motif_sub.max())

MBD2
7.546336358696746


In [7]:
### DHS ###
# Which DHSs are in the region
dhs_metadata_chr = dhs_df[dhs_df['seqname'] == region_chr]
dhs_metadata_region = dhs_metadata_chr.loc[(dhs_metadata_chr['start'] >= region_start) & (dhs_metadata_chr['end'] <= region_end)] 
# print(dhs_metadata_region)

# Component Enrichments in the region (obs/exp) # NOT LOG2 - empty values filled with 0
print(dhs_obs_over_exp.loc[region_index].iloc[3:])

# Average mean signal of DHSs
average_mean_signal = dhs_metadata_region['mean_signal'].mean()
print('avg mean_signal: ', average_mean_signal)

# Number of DHSs
print('# DHSs: ', len(dhs_metadata_region))

Cancer_epithelial        2.617097
Cardiac                  1.043255
Digestive                2.573105
Lymphoid                 0.441178
Musculoskeletal          0.569736
Myeloid_erythroid        2.653399
Neural                   0.802251
Organdevel_renal         1.553553
Placental_trophoblast    1.398956
Primitive_embryonic      0.393982
Pulmonarydevel           2.563178
Renal_cancer             3.434017
StromalA                      0.0
StromalB                 1.838517
Tissueinvariant               0.0
Vascular_endothelial     2.925656
Name: 57993, dtype: object
avg mean_signal:  0.6304765299448526
# DHSs:  37


In [8]:
### EPILOGOS ###
epilogos_chr = epilogos_states_df[epilogos_states_df['seqname'] == region_chr]
epilogos_in_region = epilogos_chr[(epilogos_chr['start'] >= region_start) & (epilogos_chr['end'] <= region_end)]
print(epilogos_in_region)


epilogos_region = epilogos_in_region.loc[:,list(range(1,19))].mean(axis=0)
# epilogos_region

epilogos_metadata_sub = epilogos_metadata[['one_index', 'short_name', 'long_name']]
epilogos_metadata_sub.set_index(epilogos_metadata_sub['one_index'], inplace=True)

epilogos_metadata_sub['score'] = epilogos_region
epilogos_metadata_sub

        seqname     start       end    1    2    3    4        5        6  \
8754190    chr3  57729600  57729800  0.0  0.0  0.0  0.0  0.00000 -0.02733   
8754191    chr3  57729800  57730000  0.0  0.0  0.0  0.0  0.00000 -0.02733   
8754192    chr3  57730000  57730200  0.0  0.0  0.0  0.0  0.00000 -0.02733   
8754193    chr3  57730200  57730400  0.0  0.0  0.0  0.0  0.00000 -0.02733   
8754194    chr3  57730400  57730600  0.0  0.0  0.0  0.0  0.00000 -0.02733   
...         ...       ...       ...  ...  ...  ...  ...      ...      ...   
8754310    chr3  57753600  57753800  0.0  0.0  0.0  0.0 -0.00957  0.50100   
8754311    chr3  57753800  57754000  0.0  0.0  0.0  0.0 -0.01050  0.52450   
8754312    chr3  57754000  57754200  0.0  0.0  0.0  0.0 -0.01050  0.51660   
8754313    chr3  57754200  57754400  0.0  0.0  0.0  0.0 -0.01191  0.49320   
8754314    chr3  57754400  57754600  0.0  0.0  0.0  0.0 -0.01128  0.48940   

                7  ...    9   10        11   12   13   14   15   16       1

/tmp/ipykernel_81848/3882697906.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  epilogos_metadata_sub['score'] = epilogos_region


,one_index,short_name,long_name,score
one_index,,,,
1,1,TssA,Active TSS,4.821679e-01
2,2,TssFlnk,Flanking TSS,1.909875e-01
3,3,TssFlnkU,Flanking TSS Upstream,1.460216e-01
4,4,TssFlnkD,Flanking TSS Downstream,1.966554e-01
5,5,Tx,Strong transcription,2.401897e-03
6,6,TxWk,Weak transcription,3.812202e-01
7,7,EnhG1,Genic Enhancer 1,1.368843e-02
8,8,EnhG2,Genic Enhancer 2,7.251040e-04
9,9,EnhA1,Active Enhancer 1,2.287642e-02


In [9]:
### GENE ###
gene_chr = gene_hg19_bed.loc[gene_hg19_bed['chr'] == region_chr]
genes_in_region = gene_chr[((gene_chr['start'] >= region_start) & (gene_chr['start'] <= region_end)) | ((gene_chr['end'] <= region_end) & (gene_chr['end'] >= region_start))]
print(genes_in_region)

print(region_start, region_end)

             chr     start       end  4 strand                6      7  \
gene_names                                                               
SLMAP       chr3  57741956  57914895  3      +  ENST00000295951  16643   

                         8                     descriptor        10        11  \
gene_names                                                                      
SLMAP       protein-coding  sarcolemma associated protein  57743378  57913115   

                                                           12  \
gene_names                                                      
SLMAP       113,1302,148,73,37,63,96,72,141,138,169,63,60,...   

                                                           13  
gene_names                                                     
SLMAP       0,318,75153,85069,90967,93524,101503,101858,10...  
57729600 57754600


In [238]:
# Extra gene tester to see what any other genes in the region might look like in terms of DHS enrichment. 
gene2_start = gene_hg19_bed.loc['PPP1R3F'].start
gene2_end = gene_hg19_bed.loc['PPP1R3F'].end
dhs_metadata_chr = dhs_df[dhs_df['seqname'] == 'chrX']
dhs_metadata_region = dhs_metadata_chr.loc[(dhs_metadata_chr['start'] >= region_start) & (dhs_metadata_chr['end'] <= region_end)] 
print(dhs_metadata_region.component.sort_values(ascending=False))

identifier
X.380685    Vascular / endothelial
X.38053           Tissue invariant
X.380682                 Stromal B
X.380678            Renal / cancer
X.380691            Renal / cancer
X.38065      Primitive / embryonic
X.380662     Primitive / embryonic
X.38054      Primitive / embryonic
X.38059                     Neural
X.380675                    Neural
X.380695       Myeloid / erythroid
X.380688       Myeloid / erythroid
X.38064        Myeloid / erythroid
X.380668       Myeloid / erythroid
X.38056            Musculoskeletal
X.380698                  Lymphoid
X.380671                  Lymphoid
X.38062                   Lymphoid
X.38061                  Digestive
X.38058                  Digestive
X.38055                  Digestive
X.380665                 Digestive
Name: component, dtype: object


In [21]:
### STRING ###
protein_assoc_hg19 = pd.read_pickle('../data/hg19-protein-associations-detailed.pkl')
print(protein_assoc_hg19.shape)

protein_assoc_hg19.index = protein_assoc_hg19['protein1_pref']

columns = ['coexpression', 'experimental', 'database', 'textmining']
subscores = protein_assoc_hg19[columns]
print(subscores.shape)

subscores_600 = subscores.loc[subscores.max(axis=1) > 150]

(10538486, 11)
(10538486, 4)


In [95]:
gene_hg19_bed_STRING = gene_hg19_bed.loc[protein_assoc_hg19.index]
gene_hg19_bed_STRING

,chr,start,end,4,strand,6,7,8,descriptor,10,11,12,13
protein1_pref,,,,,,,,,,,,,
ARF5,chr7,127228398,127231759,6,+,ENST00000000233,658,protein-coding,ADP ribosylation factor 5,127228552,127231353,"221,81,110,72,126,493,","0,738,1140,1721,2618,2868,"
ARF5,chr7,127228398,127231759,6,+,ENST00000000233,658,protein-coding,ADP ribosylation factor 5,127228552,127231353,"221,81,110,72,126,493,","0,738,1140,1721,2618,2868,"
ARF5,chr7,127228398,127231759,6,+,ENST00000000233,658,protein-coding,ADP ribosylation factor 5,127228552,127231353,"221,81,110,72,126,493,","0,738,1140,1721,2618,2868,"
ARF5,chr7,127228398,127231759,6,+,ENST00000000233,658,protein-coding,ADP ribosylation factor 5,127228552,127231353,"221,81,110,72,126,493,","0,738,1140,1721,2618,2868,"
ARF5,chr7,127228398,127231759,6,+,ENST00000000233,658,protein-coding,ADP ribosylation factor 5,127228552,127231353,"221,81,110,72,126,493,","0,738,1140,1721,2618,2868,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
PAK4,chr19,39616473,39673456,92,+,ENST00000599386,16059,protein-coding,p21 (RAC1) activated kinase 4,39660193,39669219,"86,73,226,435,134,127,126,135,4393,","0,42396,43698,47742,49097,49478,50756,51841,52..."
PAK4,chr19,39616473,39673456,92,+,ENST00000599386,16059,protein-coding,p21 (RAC1) activated kinase 4,39660193,39669219,"86,73,226,435,134,127,126,135,4393,","0,42396,43698,47742,49097,49478,50756,51841,52..."
PAK4,chr19,39616473,39673456,92,+,ENST00000599386,16059,protein-coding,p21 (RAC1) activated kinase 4,39660193,39669219,"86,73,226,435,134,127,126,135,4393,","0,42396,43698,47742,49097,49478,50756,51841,52..."


In [13]:
print(gene_vector(region_chr, region_start, region_end, gene_hg19_bed, protein_assoc_hg19))

coexpression   NaN
experimental   NaN
database       NaN
textmining     NaN
dtype: float64


# Functions for creating data vectors
Based on all the code above, these functions are the code above but returns a vector. 
These functions were used to create a big dataframe for each data modality such that each region had a vector for each data modality

In [10]:
def tf_vector(region_chr, region_start, region_end):
    motif_enrichment_chr = motif_enrichment[motif_enrichment['seqname'] == region_chr]
    motif_enrichment_region = motif_enrichment_chr.loc[(motif_enrichment_chr['start'] >= region_start) & (motif_enrichment_chr['end'] <= region_end)] 
    
    return motif_enrichment_region

def dhs_vector(region_chr, region_start, region_end):
    # Which DHSs are in the region
    dhs_metadata_chr = dhs_df[dhs_df['seqname'] == region_chr]
    dhs_metadata_region = dhs_metadata_chr.loc[(dhs_metadata_chr['start'] >= region_start) & (dhs_metadata_chr['end'] <= region_end)] 

#     # Component Enrichments in the region (obs/exp) # NOT LOG2 - empty values filled with 0
#     dhs_obs_over_exp_chr = dhs_obs_over_exp[dhs_obs_over_exp['seqname'] == region_chr]
#     dhs_obs_over_exp_region = dhs_obs_over_exp_chr.loc[(dhs_obs_over_exp_chr['start'] >= region_start) & (dhs_obs_over_exp_chr['end'] <= region_end)] 


    # Average mean signal of DHSs
    average_mean_signal = dhs_metadata_region['mean_signal'].mean()

    # Number of DHSs
    num_dhs = len(dhs_metadata_region)

    dhs_vector = [average_mean_signal, num_dhs]
    dhs_vector.extend(dhs_obs_over_exp_region.iloc[:, 3:].T.iloc[:, 0].tolist())
    
    del dhs_metadata_chr, dhs_metadata_region, dhs_obs_over_exp_chr, dhs_obs_over_exp_region
    
    return dhs_vector

def epilogos_vector(region_chr, region_start, region_end):
    epilogos_chr = epilogos_states_df[epilogos_states_df['seqname'] == region_chr]
    epilogos_in_region = epilogos_chr[(epilogos_chr['start'] >= region_start) & (epilogos_chr['end'] <= region_end)]

    epilogos_region = epilogos_in_region.loc[:,list(range(1,19))].mean(axis=0)
    return epilogos_region

def gene_vector(region_chr, region_start, region_end, gene_hg19_bed, protein_assoc_hg19):
    
    gene_hg19_bed = gene_hg19_bed[gene_hg19_bed.index.isin(protein_assoc_hg19.index)]
    protein_assoc_hg19 = protein_assoc_hg19[protein_assoc_hg19.index.isin(gene_hg19_bed.index)]
    gene_chr = gene_hg19_bed.loc[gene_hg19_bed['chr'] == region_chr]
    genes_in_region = gene_chr[((gene_chr['start'] >= region_start) & (gene_chr['start'] <= region_end)) | ((gene_chr['end'] <= region_end) & (gene_chr['end'] >= region_start))]
    

    columns = ['coexpression', 'experimental', 'database', 'textmining']
    subscores = protein_assoc_hg19[columns]
    
    genes_in_region_assoc = subscores.loc[genes_in_region.index]
    # Get max of the subscores of gene associations
    return genes_in_region_assoc.max(axis=0)

In [5]:
## Create a bed file to use to find overlaps using bedmap


motif_enrichment = pd.read_pickle('../data/motif_log2_enrichment.pkl')
dhs_df = pd.read_pickle('../data/dhs_metadata.pkl')
epilogos_states_df = pd.read_pickle('../data/epilogos_states_df')

motif_enrichment_bed = BedTool.from_dataframe(motif_enrichment)
dhs_bed = BedTool.from_dataframe(dhs_df)
epilogos_states_bed = BedTool.from_dataframe(epilogos_states_df)

motif_enrichment_bed.saveas('../data/motif_enrichment_bed.bed')
dhs_bed.saveas('../data/dhs_bed.bed')
epilogos_states_bed.saveas('../data/epilogos_states_bed.bed')

<BedTool(../data/epilogos_states_bed.bed)>

In [119]:
dhs_ms_num = pd.DataFrame()
for i in tqdm(range(len(bins_25kb))):
    region_chr = bins_25kb.iloc[i, 0]
    region_start = bins_25kb.iloc[i, 1]
    region_end = bins_25kb.iloc[i, 2]
#     print(region_chr, region_start, region_end)

    new_vector = dhs_vector(region_chr, region_start, region_end)
    dhs_ms_num[i] = new_vector
dhs_ms_num 

  0%|▏                                                                                                                                                                                                                  | 100/99996 [00:21<5:54:52,  4.69it/s]/tmp/ipykernel_10193/586572900.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  dhs_ms_num[i] = new_vector
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 99996/99996 [5:48:35<00:00,  4.78it/s]


,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,0.129388,0.176642,0.167473,0.148034,0.079265,0.203499,0.498372,0.176563,NaN,NaN,...,0.368483,0.202598,0.474190,0.445144,0.745700,0.616913,0.497687,0.320222,0.340393,0.360654
1,1.000000,2.000000,4.000000,6.000000,3.000000,1.000000,5.000000,7.000000,0.0,0.0,...,13.000000,13.000000,29.000000,19.000000,28.000000,30.000000,26.000000,20.000000,17.000000,13.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.654274,0.0,0.0,...,0.654274,1.308548,1.308548,0.000000,1.308548,0.654274,1.308548,1.962823,0.654274,3.271371
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,1.043255,5.216275,1.043255,0.000000,2.086510,1.043255,0.000000,1.043255
4,0.000000,0.000000,0.857702,0.857702,0.857702,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,1.715403,0.857702,1.715403,0.857702,0.857702,0.000000
5,0.000000,0.000000,0.000000,0.441178,0.441178,0.000000,0.882355,0.000000,0.0,0.0,...,0.000000,0.882355,3.529421,0.441178,0.882355,0.882355,0.882355,0.441178,0.000000,0.882355
6,0.000000,0.000000,0.000000,0.569736,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.569736,2.278942,0.569736,1.139471,0.569736,0.569736,0.569736,0.569736
7,0.000000,0.000000,0.000000,0.000000,0.000000,0.663350,0.000000,0.663350,0.0,0.0,...,1.326700,0.663350,0.663350,1.326700,1.990049,1.990049,2.653399,0.663350,2.653399,0.000000
8,0.000000,0.267417,0.000000,0.267417,0.000000,0.000000,0.000000,0.534834,0.0,0.0,...,1.069668,1.069668,0.534834,0.534834,0.267417,0.802251,0.267417,0.267417,0.267417,0.267417
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.776777,1.553553,0.000000,0.000000,0.776777


In [3]:
dhs_ms_num = pd.read_pickle('../data/dhs_vector_25kb.pkl')
dhs_ms_num
# dhs_ms_num.T.to_pickle('../../../tmp/data/dhs_vector_25kb.pkl')

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,0.129388,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.785656,0.000000
1,0.176642,2.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.267417,0.000000,0.466319,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000
2,0.167473,4.0,0.000000,0.000000,0.857702,0.000000,0.000000,0.000000,0.000000,0.000000,0.932638,0.196991,0.0,0.000000,0.0,0.000000,0.000000,0.000000
3,0.148034,6.0,0.000000,0.000000,0.857702,0.441178,0.569736,0.000000,0.267417,0.000000,0.932638,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000
4,0.079265,3.0,0.000000,0.000000,0.857702,0.441178,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.306419,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.616913,30.0,0.654274,0.000000,0.857702,0.882355,1.139471,1.990049,0.802251,0.776777,0.000000,0.393982,0.0,6.868034,0.0,1.225678,1.571313,1.462828
99992,0.497687,26.0,1.308548,2.086510,1.715403,0.882355,0.569736,2.653399,0.267417,1.553553,0.932638,0.590973,0.0,0.000000,0.0,0.612839,0.785656,2.925656
99993,0.320222,20.0,1.962823,1.043255,0.857702,0.441178,0.569736,0.663350,0.267417,0.000000,0.000000,0.196991,0.0,1.717009,0.0,1.532097,1.571313,1.462828
99994,0.340393,17.0,0.654274,0.000000,0.857702,0.000000,0.569736,2.653399,0.267417,0.000000,0.466319,0.787964,0.0,1.717009,0.0,0.306419,0.000000,1.462828


In [87]:
epilogos_vectors = pd.DataFrame()

for i in tqdm(range(len(bins_25kb))):
    region_chr = bins_25kb.iloc[i, 0]
    region_start = bins_25kb.iloc[i, 1]
    region_end = bins_25kb.iloc[i, 2]
#     print(region_chr, region_start, region_end)

    new_vector = epilogos_vector(region_chr, region_start, region_end)
    epilogos_vectors[i] = new_vector
    
epilogos_vectors.to_pickle('../data/epilogos_vector_25kb.pkl')
epilogos_vectors


  0%|▏                                                                                                                                                                                                               | 100/99996 [01:34<26:23:54,  1.05it/s]/tmp/ipykernel_1144/647887236.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  epilogos_vectors[i] = new_vector
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 99996/99996 [24:10:15<00:00,  1.15it/s]


,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
1,0.000000,0.000000,0.000000,0.000000,-0.000039,-0.000019,0.00,0.00,0.000000,0.000000,...,0.000000,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,0.000156,0.000000,0.000000,0.000000,0.000000,-0.000015,0.00,0.00,0.000000,-0.000008,...,0.000000,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.00,0.000000,0.000000,...,0.000000,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,0.000016,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.00,0.000000,0.000000,...,0.000000,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,-0.000124,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.00,-0.000083,-0.000083,...,0.000000,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
6,-0.013738,0.000000,-0.002037,-0.002095,-0.004851,-0.002735,0.00,0.00,-0.001571,-0.006635,...,0.000000,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
7,-0.000005,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.00,0.000000,0.000000,...,0.000000,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.00,0.000000,0.000000,...,0.000000,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.00,0.000000,0.000000,...,0.000000,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
10,-0.000082,0.000000,0.000000,0.000000,-0.000048,0.000000,0.00,0.00,0.000000,0.000000,...,0.000000,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [6]:
# epilogos_vectors.T.to_pickle('../data/epilogos_vector_25kb.pkl')

epilogos_vector = pd.read_pickle('../data/epilogos_vector_25kb.pkl')
epilogos_cosine = cosine_similarity(epilogos_vector)
epilogos_cosine = pd.DataFrame(epilogos_cosine)
epilogos_cosine

In [7]:
dhs_ms_num = pd.read_pickle('../data/dhs_vector_25kb.pkl')
dhs_ms_num_0fill = dhs_ms_num
dhs_ms_num_0fill[0] = dhs_ms_num_0fill[0].fillna(0)
dhs_ms_num_0fill

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,0.129388,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.785656,0.000000
1,0.176642,2.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.267417,0.000000,0.466319,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000
2,0.167473,4.0,0.000000,0.000000,0.857702,0.000000,0.000000,0.000000,0.000000,0.000000,0.932638,0.196991,0.0,0.000000,0.0,0.000000,0.000000,0.000000
3,0.148034,6.0,0.000000,0.000000,0.857702,0.441178,0.569736,0.000000,0.267417,0.000000,0.932638,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000
4,0.079265,3.0,0.000000,0.000000,0.857702,0.441178,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.306419,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.616913,30.0,0.654274,0.000000,0.857702,0.882355,1.139471,1.990049,0.802251,0.776777,0.000000,0.393982,0.0,6.868034,0.0,1.225678,1.571313,1.462828
99992,0.497687,26.0,1.308548,2.086510,1.715403,0.882355,0.569736,2.653399,0.267417,1.553553,0.932638,0.590973,0.0,0.000000,0.0,0.612839,0.785656,2.925656
99993,0.320222,20.0,1.962823,1.043255,0.857702,0.441178,0.569736,0.663350,0.267417,0.000000,0.000000,0.196991,0.0,1.717009,0.0,1.532097,1.571313,1.462828
99994,0.340393,17.0,0.654274,0.000000,0.857702,0.000000,0.569736,2.653399,0.267417,0.000000,0.466319,0.787964,0.0,1.717009,0.0,0.306419,0.000000,1.462828


In [8]:
dhs_cosine = cosine_similarity(dhs_ms_num_0fill)

In [84]:
dhs_ms_num_0fill

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17
0,0.129388,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.785656,0.000000
1,0.176642,2.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.267417,0.000000,0.466319,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000
2,0.167473,4.0,0.000000,0.000000,0.857702,0.000000,0.000000,0.000000,0.000000,0.000000,0.932638,0.196991,0.0,0.000000,0.0,0.000000,0.000000,0.000000
3,0.148034,6.0,0.000000,0.000000,0.857702,0.441178,0.569736,0.000000,0.267417,0.000000,0.932638,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000
4,0.079265,3.0,0.000000,0.000000,0.857702,0.441178,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.306419,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,0.616913,30.0,0.654274,0.000000,0.857702,0.882355,1.139471,1.990049,0.802251,0.776777,0.000000,0.393982,0.0,6.868034,0.0,1.225678,1.571313,1.462828
99992,0.497687,26.0,1.308548,2.086510,1.715403,0.882355,0.569736,2.653399,0.267417,1.553553,0.932638,0.590973,0.0,0.000000,0.0,0.612839,0.785656,2.925656
99993,0.320222,20.0,1.962823,1.043255,0.857702,0.441178,0.569736,0.663350,0.267417,0.000000,0.000000,0.196991,0.0,1.717009,0.0,1.532097,1.571313,1.462828
99994,0.340393,17.0,0.654274,0.000000,0.857702,0.000000,0.569736,2.653399,0.267417,0.000000,0.466319,0.787964,0.0,1.717009,0.0,0.306419,0.000000,1.462828


In [10]:
motif_cosine = cosine_similarity(obs_over_exp)
motif_cosine

array([[1.        , 0.51747916, 0.43146512, ..., 0.49996729, 0.58965489,
        0.56761075],
       [0.51747916, 1.        , 0.76275737, ..., 0.78173024, 0.7679177 ,
        0.76046178],
       [0.43146512, 0.76275737, 1.        , ..., 0.79582685, 0.80124203,
        0.8352996 ],
       ...,
       [0.49996729, 0.78173024, 0.79582685, ..., 1.        , 0.83712506,
        0.84080376],
       [0.58965489, 0.7679177 , 0.80124203, ..., 0.83712506, 1.        ,
        0.88868056],
       [0.56761075, 0.76046178, 0.8352996 , ..., 0.84080376, 0.88868056,
        1.        ]])

In [60]:
motif_counts_df = pd.read_pickle('../data/motif_counts_df_25kb.pkl')
motif_counts_df.iloc[:, 3:] = motif_counts_df.iloc[:, 3:].astype(int)
motif_counts_df.isnull().any()

seqname    False
start      False
end        False
AHR        False
AIRE       False
           ...  
ZNF768     False
ZNF784     False
ZNF85      False
ZSCAN3     False
ZSCAN4     False
Length: 285, dtype: bool

## Motif obs/exp calculation

In [61]:
bin_cols = ['seqname', 'start', 'end', '?']
bins_25kb = pd.read_csv('../data/Altius_100k_epilogos_hg19_WM20210518.bed', delimiter='\t', names=bin_cols)


motif_counts_df.iloc[:, 3:]
total_tf_cluster_counts_expected = motif_counts_df.iloc[:, 3:].sum(axis=0)/len(bins_25kb)

In [73]:
obs_over_exp = motif_counts_df.iloc[:, 3:]/total_tf_cluster_counts_expected
# obs_over_exp[obs_over_exp.columns[obs_over_exp.isna().any()]]
obs_over_exp

,AHR,AIRE,AP1_1,AP1_2,ARI5A,ARI5B,BATF,BCL6_1,BCL6_2,CCAAT_CEBP,...,ZNF652,ZNF667,ZNF680,ZNF708,ZNF713,ZNF768,ZNF784,ZNF85,ZSCAN3,ZSCAN4
0,1.919211,1.720520,0.802816,1.688585,0.506587,0.732321,0.698827,0.000000,1.654323,0.707443,...,0.794104,2.261862,1.518223,3.657961,0.000000,2.530763,1.400308,0.862471,2.585241,1.140042
1,0.000000,1.720520,1.162699,0.761519,1.013174,0.732321,1.087065,0.000000,1.024104,1.131909,...,0.794104,0.527768,0.867556,0.313539,0.992605,1.190947,0.933539,0.934343,0.982392,1.072980
2,0.639737,1.720520,0.747450,0.695300,0.759881,0.976428,0.465885,0.000000,1.024104,1.792189,...,0.794104,0.603163,1.518223,1.045132,0.992605,0.893210,0.700154,1.581196,0.465343,0.536490
3,1.599342,0.860260,1.328799,1.158833,1.013174,1.464642,1.087065,1.070655,2.048209,1.556375,...,1.588209,0.678559,1.518223,1.045132,0.000000,1.091702,0.933539,0.862471,1.085801,2.481267
4,0.000000,0.430130,0.941233,1.423709,1.773055,1.708749,0.931770,2.141311,1.102882,1.037583,...,0.529403,1.734094,1.409778,1.149645,1.488907,0.793965,1.166923,0.934343,1.706259,0.469429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,1.279474,0.645195,1.273433,1.258162,0.759881,0.732321,1.087065,2.141311,0.945327,1.179072,...,1.852910,1.206326,0.759111,1.149645,0.496302,1.091702,1.867077,0.646853,0.878982,0.670613
99992,2.239079,1.075325,0.830500,0.728409,2.026349,0.976428,1.009417,2.141311,1.181659,1.367723,...,1.058806,1.281722,1.084445,1.149645,1.488907,1.786421,0.700154,0.790598,1.499440,1.005919
99993,0.959605,1.505455,0.885866,1.125724,1.013174,0.732321,0.776475,1.070655,1.102882,1.462049,...,1.588209,0.829349,0.759111,0.836105,0.992605,0.496228,0.466769,0.790598,0.413639,0.871797
99994,1.279474,1.075325,0.830500,0.662190,1.013174,0.976428,0.543532,0.000000,1.024104,1.084746,...,1.058806,1.507908,0.759111,0.731592,1.488907,0.992456,0.933539,0.790598,0.930687,1.408287


In [9]:
# obs_over_exp.to_pickle('../data/motif_obs_over_exp.pkl')
obs_over_exp = pd.read_pickle('../data/motif_obs_over_exp.pkl')

In [24]:
motif_cosine = pd.read_pickle('../data/motif_cosine_25kb.pkl')
dhs_cosine = pd.read_pickle('../data/dhs_cosine_25kb.pkl')
epilogos_cosine = pd.read_pickle('../data/epilogos_cosine_25kb.pkl')

KeyboardInterrupt: 

In [16]:
gene_vectors = pd.DataFrame()

for i in tqdm(range(len(bins_25kb))):
# for i in tqdm(range(10)):
    region_chr = bins_25kb.iloc[i, 0]
    region_start = bins_25kb.iloc[i, 1]
    region_end = bins_25kb.iloc[i, 2]

    new_vector = gene_vector(region_chr, region_start, region_end, gene_hg19_bed, protein_assoc_hg19)
    gene_vectors[i] = new_vector
    
gene_vectors.to_pickle('../data/gene_vector_25kb.pkl')
# gene_vectors
gene_vectors

  0%|▏                                                                                                                                                                                                               | 100/99996 [01:34<23:33:37,  1.18it/s]/tmp/ipykernel_174379/2042960265.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  gene_vectors[i] = new_vector
 91%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍                 | 91375/99996 [25:07:10<2:22:11,  1.01it/s]


KeyError: "['LOC730098'] not in index"

In [12]:
gene_vectors = pd.read_pickle('../data/gene_vector_25kb.pkl')

In [17]:
gene_vectors

,0,1,2,3,4,5,6,7,8,9,...,91365,91366,91367,91368,91369,91370,91371,91372,91373,91374
coexpression,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,122.0,171.0,NaN,171.0,NaN,350.0,348.0,348.0,649.0,854.0
experimental,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,302.0,488.0,NaN,488.0,NaN,0.0,875.0,875.0,951.0,937.0
database,NaN,NaN,600.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,0.0,NaN,0.0,800.0,800.0,900.0,900.0
textmining,NaN,NaN,654.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,612.0,687.0,NaN,687.0,NaN,739.0,987.0,987.0,775.0,988.0


In [27]:
region_chr = bins_25kb.iloc[91375, 0]
region_start = bins_25kb.iloc[91375, 1]
region_end = bins_25kb.iloc[91375, 2]
gene_vector(region_chr, region_start, region_end, gene_hg19_bed, protein_assoc_hg19)

coexpression    545.0
experimental    911.0
database        800.0
textmining      989.0
dtype: float64

In [28]:
gene_vectors

,0,1,2,3,4,5,6,7,8,9,...,91365,91366,91367,91368,91369,91370,91371,91372,91373,91374
coexpression,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,122.0,171.0,NaN,171.0,NaN,350.0,348.0,348.0,649.0,854.0
experimental,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,302.0,488.0,NaN,488.0,NaN,0.0,875.0,875.0,951.0,937.0
database,NaN,NaN,600.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,NaN,0.0,NaN,0.0,800.0,800.0,900.0,900.0
textmining,NaN,NaN,654.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,612.0,687.0,NaN,687.0,NaN,739.0,987.0,987.0,775.0,988.0


In [38]:
STRING_cosine = nan_euclidean_distances(gene_vectors)
STRING_cosine.shape

(99996, 99996)

In [41]:
STRING_cosine_df = pd.DataFrame(STRING_cosine)
STRING_cosine_df

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,432.482370,NaN,NaN,NaN,1166.242256,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99992,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99993,NaN,NaN,1166.242256,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,838.379389,NaN,NaN,NaN,0.000000,NaN,NaN
99994,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
STRING_cosine_df.iloc[96548, :].sort_values()

96548    0.000000
57636    5.744563
28863    6.557439
99943    7.681146
91328    7.874008
           ...   
99990         NaN
99991         NaN
99992         NaN
99994         NaN
99995         NaN
Name: 96548, Length: 99996, dtype: float64

In [43]:
STRING_cosine_df.to_pickle('../output/STRING_cosine_25kb.pkl')

# Top 100 similar region indices based on STRING
We take the top 100 indices and set everything else to 0 in terms of scoring. This will save memory.

**This is still incomplete.**

In [ ]:
# Make the 100k by 100 array
top100idx_25kb_df = pd.DataFrame()

for i in tqdm(range(len(bins_25kb))):
    region_chr = bins_25kb.iloc[i, 0]
    region_start = bins_25kb.iloc[i, 1]
    region_end = bins_25kb.iloc[i, 2]
    
def top_k(x,k):
    ind=np.argpartition(x,-1*k)[-1*k:]
    return ind[np.argsort(x[ind])]    

In [68]:
top100idx_25kb_df = pd.DataFrame()

for i in tqdm(range(len(bins_25kb))):
#     top100idx_25kb_df[i] = STRING_cosine_df.iloc[i, :].sort_values(ascending=False).index
    top100idx_25kb_df[i] = STRING_cosine_df.iloc[i, :].sort_values(ascending=False).index

  0%|▏                                                                                                                                                                                                                  | 98/99996 [00:00<15:38, 106.45it/s]/tmp/ipykernel_174379/1521215485.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead.  To get a de-fragmented frame, use `newframe = frame.copy()`
  top100idx_25kb_df[i] = STRING_cosine_df.iloc[i, :].sort_values(ascending=False).index
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 99996/99996 [11:40<00:00, 142.78it/s]


In [69]:
top100idx_25kb_df

,0,1,2,3,4,5,6,7,8,9,...,99986,99987,99988,99989,99990,99991,99992,99993,99994,99995
0,0,0,22449,0,0,0,0,0,0,0,...,0,0,0,22449,0,0,0,86479,0,0
1,1,1,36319,1,1,1,1,1,1,1,...,1,1,1,36319,1,1,1,14622,1,1
2,2,2,62559,2,2,2,2,2,2,2,...,2,2,2,62559,2,2,2,57816,2,2
3,3,3,12900,3,3,3,3,3,3,3,...,3,3,3,12900,3,3,3,81151,3,3
4,4,4,42542,4,4,4,4,4,4,4,...,4,4,4,42542,4,4,4,32960,4,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99991,99991,99991,99990,99991,99991,99991,99991,99991,99991,99991,...,99991,99991,99991,99990,99991,99991,99991,99990,99991,99991
99992,99992,99992,99991,99992,99992,99992,99992,99992,99992,99992,...,99992,99992,99992,99991,99992,99992,99992,99991,99992,99992
99993,99993,99993,99992,99993,99993,99993,99993,99993,99993,99993,...,99993,99993,99993,99992,99993,99993,99993,99992,99993,99993
99994,99994,99994,99994,99994,99994,99994,99994,99994,99994,99994,...,99994,99994,99994,99994,99994,99994,99994,99994,99994,99994
